<a href="https://colab.research.google.com/github/betulbilhan2/ai-carbon-tracker/blob/main/NB02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 02: TabNet Feature Engineering & Mapping

Bu notebook, **TerkenTech — Yapay Zeka Pipeline'ı: Kesinleşmiş Roadmap (v4)** dokümanında belirtilen **2. Adım** işlemlerini gerçekleştirir.

**Görevler:**
1. Notebook 1'den gelen `tabnet_train.parquet`, `tabnet_val.parquet`, `tabnet_test.parquet` verilerini okumak.
2. Sayısal kolonlar için `StandardScaler` uygulamak (sadece train setine fit edilerek data leakage'ı önlemek).
3. Kategorik değişkenler için string -> integer mapping (API sözleşmesine uygun) sözlüğünü oluşturmak.
4. TabNet'in ihtiyaç duyduğu `categorical_dims` gibi bilgileri hesaplamak.
5. Tüm bu metadataları `feature_metadata.json` dosyasına yazmak ve ölçeklenmiş veri setlerini kaydetmek.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os
import json
import joblib
from sklearn.preprocessing import StandardScaler

print("Kütüphaneler başarıyla yüklendi ve Drive bağlandı!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Kütüphaneler başarıyla yüklendi ve Drive bağlandı!


In [ ]:
# 1. Verileri yükleme
base_path = '/content/drive/MyDrive/terkentech'

train = pd.read_parquet(f'{base_path}/01_processed/tabnet_train.parquet')
val = pd.read_parquet(f'{base_path}/01_processed/tabnet_val.parquet')
test = pd.read_parquet(f'{base_path}/01_processed/tabnet_test.parquet')

print(f"Train set: {train.shape}")
print(f"Val set: {val.shape}")
print(f"Test set: {test.shape}")

Train set: (7000, 20)
Val set: (1500, 20)
Test set: (1500, 20)


In [ ]:
# 2. Pipeline Roadmap'e göre kolon tiplerinin belirlenmesi
# (Etik sebeplerden ve modelin genellenebilirliği açısından Body Type ve Sex modelden çıkarılabilir,
# K-Means için çıkarılmıştı, biz de TabNet tahmininde demografik bias olmaması için feature'lardan ayırıyoruz)

target_col = 'CarbonEmission'

numerical_cols = [
    'Monthly Grocery Bill',
    'Vehicle Monthly Distance Km',
    'Waste Bag Weekly Count',
    'How Long TV PC Daily Hour',
    'How Many New Clothes Monthly',
    'How Long Internet Daily Hour'
]

categorical_cols = [
    'Diet', 'How Often Shower', 'Heating Energy Source',
    'Transport', 'Vehicle Type', 'Social Activity', 'Frequency of Traveling by Air',
    'Waste Bag Size', 'Energy efficiency', 'Recycling', 'Cooking_With'
]

features = numerical_cols + categorical_cols

print(f"Sayısal özellikler ({len(numerical_cols)}): {numerical_cols}")
print(f"Kategorik özellikler ({len(categorical_cols)}): {categorical_cols}")

Sayısal özellikler (6): ['Monthly Grocery Bill', 'Vehicle Monthly Distance Km', 'Waste Bag Weekly Count', 'How Long TV PC Daily Hour', 'How Many New Clothes Monthly', 'How Long Internet Daily Hour']
Kategorik özellikler (11): ['Diet', 'How Often Shower', 'Heating Energy Source', 'Transport', 'Vehicle Type', 'Social Activity', 'Frequency of Traveling by Air', 'Waste Bag Size', 'Energy efficiency', 'Recycling', 'Cooking_With']


In [ ]:
# 3. Sayısal kolonlar için StandardScaler (sadece train setine fit ediyoruz)
scaler = StandardScaler()

# Kopya oluşturarak ilerleyelim
train_scaled = train.copy()
val_scaled = val.copy()
test_scaled = test.copy()

train_scaled[numerical_cols] = scaler.fit_transform(train[numerical_cols])
val_scaled[numerical_cols] = scaler.transform(val[numerical_cols])
test_scaled[numerical_cols] = scaler.transform(test[numerical_cols])

# Scaler'ı kaydedelim (Deployment ve K-Means'de lazım olacak)
os.makedirs(f'{base_path}/02_models/tabnet', exist_ok=True)
joblib.dump(scaler, f'{base_path}/02_models/tabnet/standard_scaler.pkl')

print(f"Sayısal kolonlar ölçeklendi ve scaler {base_path}/02_models/tabnet/standard_scaler.pkl olarak kaydedildi.")

Sayısal kolonlar ölçeklendi ve scaler /content/drive/MyDrive/terkentech/02_models/tabnet/standard_scaler.pkl olarak kaydedildi.


In [ ]:
# 4. Kategorik Değişkenler İçin String -> Integer Mapping Artifact
# Bölüm 5 uyarınca: Mobil uygulamadan gelecek string değerlerin,
# modelin anlayacağı tam sayı (integer) kodlarına çevrilmesi için mapping.

category_mapping = {
    "diet": {"omnivore": 0, "pescatarian": 1, "vegetarian": 2, "vegan": 3},
    "how_often_shower": {"daily": 0, "twice_a_day": 1, "less_frequently": 2},
    "heating_energy_source": {"coal": 0, "natural_gas": 1, "wood": 2, "electricity": 3},
    "transport": {"public": 0, "private": 1, "walk_bicycle": 2},
    "vehicle_type": {"none": 0, "petrol": 1, "diesel": 2, "hybrid": 3, "electric": 4, "lpg": 5},
    "social_activity": {"often": 0, "sometimes": 1, "never": 2},
    "frequency_of_traveling_by_air": {"never": 0, "rarely": 1, "frequently": 2, "very_frequently": 3},
    "waste_bag_size": {"small": 0, "medium": 1, "large": 2, "extra_large": 3},
    "energy_efficiency": {"low": 0, "medium": 1, "high": 2},
    "recycling": {
        "description": "Bitmask (4-bit). Mobil 'true/false' gönderir, pipeline toplayıp tek integer yapar.",
        "paper": 1,
        "plastic": 2,
        "glass": 4,
        "metal": 8
    }
}

# TabNet için kategorik kolonların unique cardinality değerlerini bulalım
categorical_dims = {}
for col in categorical_cols:
    dim = int(train[col].max()) + 1
    categorical_dims[col] = dim

print("Kategorik kolon boyutları (TabNet embeddings için):")
print(categorical_dims)

Kategorik kolon boyutları (TabNet embeddings için):
{'Diet': 4, 'How Often Shower': 4, 'Heating Energy Source': 4, 'Transport': 3, 'Vehicle Type': 6, 'Social Activity': 3, 'Frequency of Traveling by Air': 4, 'Waste Bag Size': 4, 'Energy efficiency': 3, 'Recycling': 16, 'Cooking_With': 16}


In [ ]:
# 5. İşlenmiş dosyaları kaydetme ve feature_metadata.json güncelleme

train_scaled.to_parquet(f'{base_path}/01_processed/tabnet_train_scaled.parquet', index=False)
val_scaled.to_parquet(f'{base_path}/01_processed/tabnet_val_scaled.parquet', index=False)
test_scaled.to_parquet(f'{base_path}/01_processed/tabnet_test_scaled.parquet', index=False)

metadata_path = f'{base_path}/01_processed/feature_metadata.json'

if os.path.exists(metadata_path):
    with open(metadata_path, 'r', encoding='utf-8') as f:
        metadata = json.load(f)
else:
    metadata = {}

metadata["categorical_mapping"] = category_mapping
metadata["model_features"] = features
metadata["numerical_columns"] = numerical_cols
metadata["categorical_columns"] = categorical_cols
metadata["categorical_dims"] = categorical_dims
metadata["target_column"] = target_col
metadata["scaler_path"] = f"{base_path}/02_models/tabnet/standard_scaler.pkl"

with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=4)

print("İşlenmiş veri setleri kaydedildi: 'tabnet_train_scaled.parquet' vs.")
print("feature_metadata.json güncellendi!")
print("\nTEBRİKLER! Notebook 2 başarıyla tamamlandı. Sırada Notebook 3 (TabNet Eğitimi) var. 🚀")

İşlenmiş veri setleri kaydedildi: 'tabnet_train_scaled.parquet' vs.
feature_metadata.json güncellendi!

TEBRİKLER! Notebook 2 başarıyla tamamlandı. Sırada Notebook 3 (TabNet Eğitimi) var. 🚀
